# Deep Learning Regressor Pipeline (Heavy + Effective)
Notebook ini membangun model deep learning regressor yang relatif berat untuk data tabular:
1. Split data train/val/test = 80/10/10
2. Standardisasi fitur dan target
3. Model Deep Residual MLP + BatchNorm + Dropout
4. Callback: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
5. Simpan split, scaler PKL, model, history, hyperparameter, dan metrik

In [1]:
from pathlib import Path
from time import perf_counter
import json
import pickle
import random
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Pastikan TensorFlow tersedia
step_start = perf_counter()

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow'])
    import tensorflow as tf

process_times = {}
process_times['import_tensorflow'] = perf_counter() - step_start
print(f"Waktu import_tensorflow: {process_times['import_tensorflow']:.6f} detik")
print('TensorFlow version:', tf.__version__)


Waktu import_tensorflow: 19.822675 detik
TensorFlow version: 2.15.0


In [3]:
# Reproducibility
step_start = perf_counter()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

process_times['set_seed'] = perf_counter() - step_start
print(f"Waktu set_seed: {process_times['set_seed']:.6f} detik")

Waktu set_seed: 0.003882 detik


In [4]:
# Konfigurasi path
step_start = perf_counter()

DATA_PATH = Path('java-ash-hysplit-encoded.csv')
ARTIFACT_DIR = Path('artifacts_deep_regressor')
SPLIT_DIR = ARTIFACT_DIR / 'splits'
SCALER_DIR = ARTIFACT_DIR / 'scalers'
MODEL_DIR = ARTIFACT_DIR / 'model'
METRIC_DIR = ARTIFACT_DIR / 'metrics'

for d in [ARTIFACT_DIR, SPLIT_DIR, SCALER_DIR, MODEL_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

process_times['setup_paths'] = perf_counter() - step_start
print(f"Waktu setup_paths: {process_times['setup_paths']:.6f} detik")

Waktu setup_paths: 0.003979 detik


In [5]:
# Load data
step_start = perf_counter()

df = pd.read_csv(DATA_PATH)

process_times['load_data'] = perf_counter() - step_start
print(f"Waktu load_data: {process_times['load_data']:.6f} detik")
print('Shape data:', df.shape)
df.head()

Waktu load_data: 0.011224 detik
Shape data: (1707, 19)


,timestamp,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km,volcano_filter_Bromo,volcano_filter_Merapi,volcano_filter_Raung,volcano_filter_Semeru,volcano_filter_Slamet
0,6,3,-7.54194,110.44194,2968,1000,4.7,270.0,50,30,32.511505,614.390650,303.257335,18.304149,0,1,0,0,0
1,1,1,-7.54194,110.44194,2968,1000,8.0,135.0,51,168,27.761756,460.792988,37.901873,15.892819,0,1,0,0,0
2,3,1,-7.54194,110.44194,2968,1200,7.1,90.0,65,133,44.625714,614.214150,265.635506,23.934437,0,1,0,0,0
3,3,1,-7.54194,110.44194,2968,1500,3.2,135.0,70,160,22.998759,184.285530,288.108280,11.770659,0,1,0,0,0
4,3,1,-7.54194,110.44194,2968,2500,8.3,315.0,70,145,39.159595,307.107075,270.462663,20.809694,0,1,0,0,0


In [6]:
# Pilih target multi-output
step_start = perf_counter()

target_cols = ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']
missing_targets = [c for c in target_cols if c not in df.columns]
if missing_targets:
    raise ValueError(f'Target tidak ditemukan: {missing_targets}')

feature_cols = [c for c in df.columns if c not in target_cols]
X = df[feature_cols].copy()
y = df[target_cols].copy()

process_times['prepare_features_targets'] = perf_counter() - step_start
print(f"Waktu prepare_features_targets: {process_times['prepare_features_targets']:.6f} detik")
print('Jumlah fitur:', len(feature_cols))
print('Target:', target_cols)

Waktu prepare_features_targets: 0.004270 detik
Jumlah fitur: 15
Target: ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']


In [7]:
# Split 80/10/10
step_start = perf_counter()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED
)

process_times['split_data'] = perf_counter() - step_start
print(f"Waktu split_data: {process_times['split_data']:.6f} detik")
print('X_train:', X_train.shape, '| y_train:', y_train.shape)
print('X_val:  ', X_val.shape, '| y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, '| y_test: ', y_test.shape)

Waktu split_data: 0.007676 detik
X_train: (1365, 15) | y_train: (1365, 4)
X_val:   (171, 15) | y_val:   (171, 4)
X_test:  (171, 15) | y_test:  (171, 4)


In [8]:
# Standardisasi
step_start = perf_counter()

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = pd.DataFrame(x_scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(x_scaler.transform(X_val), columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(x_scaler.transform(X_test), columns=feature_cols, index=X_test.index)

y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=target_cols, index=y_train.index)
y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=target_cols, index=y_val.index)
y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=target_cols, index=y_test.index)

process_times['standardize_data'] = perf_counter() - step_start
print(f"Waktu standardize_data: {process_times['standardize_data']:.6f} detik")

Waktu standardize_data: 0.033900 detik


In [9]:
# Simpan split data (raw + scaled)
step_start = perf_counter()

X_train.to_csv(SPLIT_DIR / 'X_train_raw.csv', index=False)
X_val.to_csv(SPLIT_DIR / 'X_val_raw.csv', index=False)
X_test.to_csv(SPLIT_DIR / 'X_test_raw.csv', index=False)
y_train.to_csv(SPLIT_DIR / 'y_train_raw.csv', index=False)
y_val.to_csv(SPLIT_DIR / 'y_val_raw.csv', index=False)
y_test.to_csv(SPLIT_DIR / 'y_test_raw.csv', index=False)

X_train_scaled.to_csv(SPLIT_DIR / 'X_train_scaled.csv', index=False)
X_val_scaled.to_csv(SPLIT_DIR / 'X_val_scaled.csv', index=False)
X_test_scaled.to_csv(SPLIT_DIR / 'X_test_scaled.csv', index=False)
y_train_scaled.to_csv(SPLIT_DIR / 'y_train_scaled.csv', index=False)
y_val_scaled.to_csv(SPLIT_DIR / 'y_val_scaled.csv', index=False)
y_test_scaled.to_csv(SPLIT_DIR / 'y_test_scaled.csv', index=False)

process_times['save_splits'] = perf_counter() - step_start
print(f"Waktu save_splits: {process_times['save_splits']:.6f} detik")

Waktu save_splits: 0.094470 detik


In [10]:
# Simpan scaler PKL + metadata
step_start = perf_counter()

with open(SCALER_DIR / 'x_scaler.pkl', 'wb') as f:
    pickle.dump(x_scaler, f)
with open(SCALER_DIR / 'y_scaler.pkl', 'wb') as f:
    pickle.dump(y_scaler, f)

x_scaler_metadata = {
    'feature_names': feature_cols,
    'mean': x_scaler.mean_.tolist(),
    'scale': x_scaler.scale_.tolist(),
    'var': x_scaler.var_.tolist(),
    'n_features_in': int(x_scaler.n_features_in_),
}
y_scaler_metadata = {
    'target_names': target_cols,
    'mean': y_scaler.mean_.tolist(),
    'scale': y_scaler.scale_.tolist(),
    'var': y_scaler.var_.tolist(),
    'n_features_in': int(y_scaler.n_features_in_),
}

with open(SCALER_DIR / 'x_scaler_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(x_scaler_metadata, f, indent=2)
with open(SCALER_DIR / 'y_scaler_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(y_scaler_metadata, f, indent=2)

process_times['save_scalers_and_metadata'] = perf_counter() - step_start
print(f"Waktu save_scalers_and_metadata: {process_times['save_scalers_and_metadata']:.6f} detik")

Waktu save_scalers_and_metadata: 0.004045 detik


In [11]:
# Konversi ke numpy untuk training TensorFlow
X_train_np = X_train_scaled.values.astype('float32')
X_val_np = X_val_scaled.values.astype('float32')
X_test_np = X_test_scaled.values.astype('float32')

y_train_np = y_train_scaled.values.astype('float32')
y_val_np = y_val_scaled.values.astype('float32')
y_test_np = y_test_scaled.values.astype('float32')

input_dim = X_train_np.shape[1]
output_dim = y_train_np.shape[1]
print('input_dim:', input_dim, '| output_dim:', output_dim)

input_dim: 15 | output_dim: 4


In [12]:
# Hyperparameter model deep learning (heavy)
dl_params = {
    'hidden_units': [1024, 1024, 512, 512, 256],
    'dropout_rate': 0.30,
    'activation': 'swish',
    'batch_size': 64,
    'epochs': 500,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'patience_early_stopping': 40,
    'patience_reduce_lr': 15,
    'reduce_lr_factor': 0.5,
    'min_lr': 1e-6
}
dl_params

{'hidden_units': [1024, 1024, 512, 512, 256],
 'dropout_rate': 0.3,
 'activation': 'swish',
 'batch_size': 64,
 'epochs': 500,
 'learning_rate': 0.001,
 'weight_decay': 1e-05,
 'patience_early_stopping': 40,
 'patience_reduce_lr': 15,
 'reduce_lr_factor': 0.5,
 'min_lr': 1e-06}

In [13]:
# Definisi model: Deep Residual MLP
def build_deep_residual_regressor(input_dim, output_dim, params):
    inputs = tf.keras.Input(shape=(input_dim,), name='features')

    x = tf.keras.layers.Dense(params['hidden_units'][0])(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(params['activation'])(x)
    x = tf.keras.layers.Dropout(params['dropout_rate'])(x)

    for units in params['hidden_units'][1:]:
        skip = x
        x = tf.keras.layers.Dense(units)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation(params['activation'])(x)
        x = tf.keras.layers.Dropout(params['dropout_rate'])(x)

        if int(skip.shape[-1]) != units:
            skip = tf.keras.layers.Dense(units)(skip)

        x = tf.keras.layers.Add()([x, skip])

    outputs = tf.keras.layers.Dense(output_dim, name='regression_output')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='deep_residual_regressor')

    try:
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=params['learning_rate'],
            weight_decay=params['weight_decay']
        )
    except AttributeError:
        optimizer = tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.Huber(),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae'), tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    return model

In [14]:
# Training model
step_start = perf_counter()

model = build_deep_residual_regressor(input_dim, output_dim, dl_params)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=dl_params['patience_early_stopping'],
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=dl_params['reduce_lr_factor'],
        patience=dl_params['patience_reduce_lr'],
        min_lr=dl_params['min_lr']
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_model.keras'),
        monitor='val_loss',
        save_best_only=True
    )
]

history = model.fit(
    X_train_np, y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=dl_params['epochs'],
    batch_size=dl_params['batch_size'],
    callbacks=callbacks,
    verbose=1
)

model.save(MODEL_DIR / 'final_model.keras')

history_df = pd.DataFrame(history.history)
history_df.to_csv(METRIC_DIR / 'training_history.csv', index=False)

with open(MODEL_DIR / 'dl_hyperparameters.json', 'w', encoding='utf-8') as f:
    json.dump(dl_params, f, indent=2)

process_times['train_model'] = perf_counter() - step_start
print(f"Waktu train_model: {process_times['train_model']:.6f} detik")
model.summary()


Epoch 1/500

22/22 [==============================] - 7s 96ms/step - loss: 1.6763 - mae: 2.0796 - rmse: 4.7516 - val_loss: 0.3013 - val_mae: 0.6071 - val_rmse: 1.0307 - lr: 0.0010
Epoch 2/500
22/22 [==============================] - 1s 63ms/step - loss: 0.7146 - mae: 1.0742 - rmse: 2.1878 - val_loss: 0.2895 - val_mae: 0.5917 - val_rmse: 1.0028 - lr: 0.0010
Epoch 3/500
22/22 [==============================] - 2s 71ms/step - loss: 0.5721 - mae: 0.9048 - rmse: 1.8774 - val_loss: 0.3000 - val_mae: 0.6209 - val_rmse: 1.0112 - lr: 0.0010
Epoch 4/500
22/22 [==============================] - 2s 74ms/step - loss: 0.5388 - mae: 0.8691 - rmse: 1.5917 - val_loss: 0.2860 - val_mae: 0.5985 - val_rmse: 0.9792 - lr: 0.0010
Epoch 5/500
22/22 [==============================] - 2s 79ms/step - loss: 0.4476 - mae: 0.7623 - rmse: 1.4850 - val_loss: 0.2834 - val_mae: 0.5987 - val_rmse: 0.9905 - lr: 0.0010
Epoch 6/500
22/22 [==============================] - 2s 73ms/step - loss: 0.4632 - mae: 0.7788 - rmse: 

In [15]:
# Evaluasi val dan test di skala asli target
step_start = perf_counter()

def evaluate_split(split_name, X_np, y_scaled_true, y_true_raw_df):
    y_pred_scaled = model.predict(X_np, verbose=0)
    y_pred_raw = y_scaler.inverse_transform(y_pred_scaled)

    rows = []
    for i, target in enumerate(target_cols):
        y_true = y_true_raw_df[target].values
        y_pred = y_pred_raw[:, i]
        rows.append({
            'split': split_name,
            'target': target,
            'MAE': float(mean_absolute_error(y_true, y_pred)),
            'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'R2': float(r2_score(y_true, y_pred))
        })
    return pd.DataFrame(rows)

val_metrics = evaluate_split('val', X_val_np, y_val_np, y_val)
test_metrics = evaluate_split('test', X_test_np, y_test_np, y_test)
metrics_df = pd.concat([val_metrics, test_metrics], ignore_index=True)
summary_df = metrics_df.groupby('split')[['MAE', 'RMSE', 'R2']].mean().reset_index()

metrics_df.to_csv(METRIC_DIR / 'metrics_per_target.csv', index=False)
summary_df.to_csv(METRIC_DIR / 'metrics_summary.csv', index=False)

process_times['evaluate_and_save_metrics'] = perf_counter() - step_start
print(f"Waktu evaluate_and_save_metrics: {process_times['evaluate_and_save_metrics']:.6f} detik")
display(metrics_df)
display(summary_df)

Waktu evaluate_and_save_metrics: 0.560233 detik


,split,target,MAE,RMSE,R2
0,val,jarak_km,7.307504,13.177544,0.613312
1,val,luas_km2,177.904454,282.245778,0.423028
2,val,sudut_deg,77.077596,99.383944,0.199999
3,val,radius_km,3.791406,7.153258,0.611305
4,test,jarak_km,7.513024,13.483043,0.771280
5,test,luas_km2,213.639647,352.891163,0.435331
6,test,sudut_deg,73.392933,95.211050,0.188713
7,test,radius_km,4.072377,7.646308,0.763843


,split,MAE,RMSE,R2
0,test,74.654495,117.307891,0.539792
1,val,66.520240,100.490131,0.461911


In [16]:
# Rekap artefak dan waktu proses
step_start = perf_counter()

saved_files = sorted([str(p) for p in ARTIFACT_DIR.rglob('*') if p.is_file()])
print('Daftar file artefak:')
for f in saved_files:
    print('-', f)

process_times['list_artifacts'] = perf_counter() - step_start
timing_df = pd.DataFrame([{'process': k, 'duration_seconds': v} for k, v in process_times.items()])
timing_df = timing_df.sort_values('duration_seconds', ascending=False).reset_index(drop=True)
timing_df.to_csv(METRIC_DIR / 'process_times.csv', index=False)
print('Rekap waktu proses disimpan di:', METRIC_DIR / 'process_times.csv')
display(timing_df)

Daftar file artefak:
- artifacts_deep_regressor\metrics\metrics_per_target.csv
- artifacts_deep_regressor\metrics\metrics_summary.csv
- artifacts_deep_regressor\metrics\training_history.csv
- artifacts_deep_regressor\model\best_model.keras
- artifacts_deep_regressor\model\dl_hyperparameters.json
- artifacts_deep_regressor\model\final_model.keras
- artifacts_deep_regressor\scalers\x_scaler.pkl
- artifacts_deep_regressor\scalers\x_scaler_metadata.json
- artifacts_deep_regressor\scalers\y_scaler.pkl
- artifacts_deep_regressor\scalers\y_scaler_metadata.json
- artifacts_deep_regressor\splits\X_test_raw.csv
- artifacts_deep_regressor\splits\X_test_scaled.csv
- artifacts_deep_regressor\splits\X_train_raw.csv
- artifacts_deep_regressor\splits\X_train_scaled.csv
- artifacts_deep_regressor\splits\X_val_raw.csv
- artifacts_deep_regressor\splits\X_val_scaled.csv
- artifacts_deep_regressor\splits\y_test_raw.csv
- artifacts_deep_regressor\splits\y_test_scaled.csv
- artifacts_deep_regressor\splits\y_

,process,duration_seconds
0,train_model,163.507261
1,import_tensorflow,19.822675
2,evaluate_and_save_metrics,0.560233
3,save_splits,0.094470
4,standardize_data,0.033900
5,load_data,0.011224
6,split_data,0.007676
7,list_artifacts,0.006466
8,prepare_features_targets,0.004270
9,save_scalers_and_metadata,0.004045
